In [1]:
import librosa
import numpy as np
import pandas as pd

In [2]:
file_path = '../data/genres_original/rock/rock.00000.wav'

y, sr = librosa.load(file_path, duration=30)  # Load the audio file with a duration of 30 seconds

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print(f"LOADED AUDIO FILE: {file_path}")
print(f"Duration: {len(y)/sr:.2f} seconds at {sr} Hz sample rate")

LOADED AUDIO FILE: ../data/genres_original/rock/rock.00000.wav
Duration: 30.00 seconds at 22050 Hz sample rate


In [4]:
chroma = librosa.feature.chroma_stft(y=y, sr=sr)
print(f"Chroma feature shape: {chroma.shape}")

chroma_mean = np.mean(chroma, axis=1)
print("Chroma value per note:")
notes = ["C","C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
for note,val in zip(notes, chroma_mean):
    bar = '█' * int(val * 50)  
    print(f"{note}: {val:.2f} {bar}")

Chroma feature shape: (12, 1292)
Chroma value per note:
C: 0.38 ██████████████████
C#: 0.49 ████████████████████████
D: 0.50 ████████████████████████
D#: 0.42 ████████████████████
E: 0.48 ███████████████████████
F: 0.33 ████████████████
F#: 0.28 ██████████████
G: 0.33 ████████████████
G#: 0.39 ███████████████████
A: 0.50 █████████████████████████
A#: 0.29 ██████████████
B: 0.27 █████████████


In [5]:
# Krumhansl-Schmuckler key profiles
# These are research-validated weights for how each note
# appears in a major vs minor key
# C profile
major_profile = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09,
                           2.52, 5.19, 2.39, 3.66, 2.29, 2.88])

minor_profile = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53,
                           2.54, 4.75, 3.98, 2.69, 3.34, 3.17])

def detect_key(chroma_mean):
    best_score = -1
    best_key = ""

    notes = ["C","C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

    for i in range(12):
        # Rotate profiles for each key
        major_rotated = np.roll(major_profile, i)
        minor_rotated = np.roll(minor_profile, i)

        # Calculate correlation scores
        major_score = np.corrcoef(chroma_mean, major_rotated)[0, 1]
        minor_score = np.corrcoef(chroma_mean, minor_rotated)[0, 1]

        if major_score > best_score:
            best_score = major_score
            best_key = notes[i] + " Major"
        if minor_score > best_score:
            best_score = minor_score
            best_key = notes[i] + " Minor"

    return best_key, round(best_score, 2)

In [6]:
key, confidence = detect_key(chroma_mean)
print(f"Detected Key: {key} (Confidence: {confidence})")

Detected Key: A Major (Confidence: 0.68)


In [8]:
y2, sr2 = librosa.load("../data/genres_original/classical/classical.00000.wav", duration=30)
chroma2 = librosa.feature.chroma_cqt(y=y2, sr=sr2)
chroma_mean2 = np.mean(chroma2, axis=1)
key2, conf2 = detect_key(chroma_mean2)
print(f"Classical — Key: {key2}, Confidence: {conf2}")

Classical — Key: D Major, Confidence: 0.82


In [25]:
def extract_energy(y, sr):
    """
    Computes energy score from 4 complementary audio features.
    Uses per-feature normalization based on realistic audio ranges.
    Returns a 0-1 score.
    """
    
    # 1. RMS — average loudness
    rms = float(np.mean(librosa.feature.rms(y=y)))
    
    # 2. Spectral centroid — brightness
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    
    # 3. Spectral rolloff — where does high frequency energy drop off?
    rolloff = float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85)))
    
    # 4. Onset strength — how punchy/rhythmically intense?
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    onset_mean = float(np.mean(onset_env))
    
    # Normalize each feature using realistic observed ranges
    # These ranges are based on typical music audio — more robust than fixed divisors
    rms_norm     = np.clip((rms - 0.005) / (0.35 - 0.005), 0, 1)
    cent_norm    = np.clip((centroid - 500) / (8000 - 500), 0, 1)
    rolloff_norm = np.clip((rolloff - 1000) / (16000 - 1000), 0, 1)
    onset_norm   = np.clip((onset_mean - 0.5) / (10.0 - 0.5), 0, 1)
    
    # Weighted combination — weights reflect how much each feature
    # contributes to perceived energy
    energy = (
        0.35 * rms_norm      +   # loudness is biggest factor
        0.25 * onset_norm    +   # punchiness second
        0.25 * cent_norm     +   # brightness third
        0.15 * rolloff_norm      # rolloff adds nuance
    )
    
    return round(float(energy), 3)

energy = extract_energy(y, sr)
print(f"Energy Score: {energy}")

energy2 = extract_energy(y2, sr2)
print(f"Classical Energy Score: {energy2}")

Energy Score: 0.223
Classical Energy Score: 0.098


In [ ]:
# Complete Camelot Wheel Mapping
CAMELOT_MAP = {
    # Major keys (B = outer ring)
    'C Major':  '8B',
    'C# Major': '3B',
    'D Major':  '10B',
    'D# Major': '5B',
    'E Major':  '12B',
    'F Major':  '7B',
    'F# Major': '2B',
    'G Major':  '9B',
    'G# Major': '4B',
    'A Major':  '11B',
    'A# Major': '6B',
    'B Major':  '1B',

    # Minor keys (A = inner ring)
    'C Minor':  '5A',
    'C# Minor': '12A',
    'D Minor':  '7A',
    'D# Minor': '2A',
    'E Minor':  '9A',
    'F Minor':  '4A',
    'F# Minor': '11A',
    'G Minor':  '6A',
    'G# Minor': '1A',
    'A Minor':  '8A',
    'A# Minor': '3A',
    'B Minor':  '10A'
}

def camelot_key(key):
    return CAMELOT_MAP.get(key, "Unknown")

camelot = camelot_key(key)
print(f"Key: {key} → Camelot: {camelot}")

Key: A Major → Camelot: 11B


In [20]:
def camelot_compatibility(key1, key2):
    """
    Returns compatibility score 0-100 between two Camelot codes.
    100 = perfect match
    75  = one step away (smooth transition)
    50  = relative major/minor swap
    0   = incompatible (clash)
    """
    if key1 == "Unknown" or key2 == "Unknown":
        return 0  # Can't determine compatibility

    num_1 = int(key1[:-1])
    type_1 = key1[-1]
    num_2 = int(key2[:-1])
    type_2 = key2[-1]

    #clockwise distance
    distance = min(abs(num_1 - num_2), 12 - abs(num_1 - num_2))

    if distance == 0 and type_1 == type_2:
        return 100  # Perfect match
    
    if distance == 1 and type_1 == type_2:
        return 75  # One step away
    
    if distance == 0 and type_1 != type_2:
        return 50  # Relative major/minor swap
    
    return 0  # Incompatible
print(f"Rock key camelot: {camelot}")

# Load a second track and check compatibility
y3, sr3 = librosa.load("../data/genres_original/rock/rock.00010.wav", duration=30)
chroma3   = librosa.feature.chroma_cqt(y=y3, sr=sr3)
key3, _   = detect_key(np.mean(chroma3, axis=1))
camelot3  = camelot_key(key3)

print(f"Rock track 2 key: {key3} → {camelot3}")
print(f"Compatibility score: {camelot_compatibility(camelot, camelot3)}/100")


Rock key camelot: 11B
Rock track 2 key: D Major → 10B
Compatibility score: 75/100


In [23]:
def analyze_track(file_path):
    """
    Analyzes a single audio file and returns its DJ-relevant features.
    
    Returns a dictionary with:
    - filename, bpm, key, camelot code, energy score
    """

    # load the audio file
    y, sr = librosa.load(file_path, duration=30)

    # BPM
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)

    # Key Detection
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = np.mean(chroma, axis=1)
    key, confidence = detect_key(chroma_mean)

    # Camelot code
    camelot = camelot_key(key)

    # Energy score
    energy = extract_energy(y, sr)

    return {
        "filename": file_path,
        "bpm": round(float(tempo[0]), 2),
        "key": key,
        "camelot": camelot,
        "energy": energy,
        "key_confidence": confidence
    }

In [26]:
test_files = [
    "../data/genres_original/rock/rock.00000.wav",
    "../data/genres_original/classical/classical.00000.wav",
    "../data/genres_original/disco/disco.00000.wav",
]

for f in test_files:
    result = analyze_track(f)
    print(f"\n{result['filename']}:")
    print(f"  BPM:        {result['bpm']}")
    print(f"  Key:        {result['key']} (confidence: {result['key_confidence']})")
    print(f"  Camelot:    {result['camelot']}")
    print(f"  Energy:     {result['energy']}")


../data/genres_original/rock/rock.00000.wav:
  BPM:        123.05
  Key:        A Major (confidence: 0.68)
  Camelot:    11B
  Energy:     0.223

../data/genres_original/classical/classical.00000.wav:
  BPM:        95.7
  Key:        D Major (confidence: 0.81)
  Camelot:    10B
  Energy:     0.098

../data/genres_original/disco/disco.00000.wav:
  BPM:        117.45
  Key:        D Major (confidence: 0.52)
  Camelot:    10B
  Energy:     0.274
